###Setup

In [ ]:
!pip install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 3.7 MB/s eta 0:00:00


In [ ]:
# Set Groq API token
# In Prod should be done through .env files
import os
os.environ['GROQ_API_KEY'] = "<>"

###Define Tools

In [ ]:
# Create a weather function/tool
def get_weather(city: str) -> int:
  """Provide weather information with city as input and temperature as output in degree celsius (integer)"""

  #api call
  res = 20

  return res


In [ ]:
# Suggest Activity
def suggest_activity(temperature: int) -> str:
  """Suggests activity to do as per Temperature as input"""

  # Logic

  return "Jump on Bed"

In [ ]:
# Create all TOOLS

TOOLS = [
    {
      "type": "function",
      "function": {
          "name": "get_weather",
          "description": (
              "Allows to get weather of a City in degree celsius"
          ),
          "parameters": {
              "type": "object",
              "properties": {
                  "city": {
                      "type": "string",
                      "description": "The name of the city",
                  }
              },
              "required": ["city"],
          },
      },
    },
    {
      "type": "function",
      "function": {
          "name": "suggest_activity",
          "description": (
              "Suggests activity to do as per Temperature as input"
          ),
          "parameters": {
              "type": "object",
              "properties": {
                  "temperature": {
                      "type": "integer",
                      "description": "Temperature in degree celsius",
                  }
              },
              "required": ["temperature"],
          },
      },
    }
]


In [ ]:
# Tool Functions - maps strings to actual functions
TOOL_FUNCTIONS = {
    "get_weather" :get_weather,
    "suggest_activity": suggest_activity
}

###ReAct Loop (Tool Calling Loop)

In [ ]:
# Question
QUESTION = "How is the weather in Bangalore?"

In [ ]:
# Create messages
MESSAGES = [
    {"role": "user", "content": QUESTION}
    ]

In [ ]:
# Call LLM with TOOL
from groq import Groq
client = Groq()
MODEL = "llama-3.1-8b-instant"

res = client.chat.completions.create(
    model = MODEL,
    max_tokens = 1000,
    tools= TOOLS,
    messages= MESSAGES
)




In [ ]:
res.choices[0]

Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content=None, role='assistant', annotations=None, executed_tools=None, function_call=None, reasoning=None, tool_calls=[ChatCompletionMessageToolCall(id='dcq8w3g8h', function=Function(arguments='{"city":"Bangalore"}', name='get_weather'), type='function')]))

In [ ]:
MESSAGES.append(res.choices[0].message)

In [ ]:
MESSAGES

[{'role': 'user', 'content': 'How is the weather in Bangalore?'},
 ChatCompletionMessage(content=None, role='assistant', annotations=None, executed_tools=None, function_call=None, reasoning=None, tool_calls=[ChatCompletionMessageToolCall(id='dcq8w3g8h', function=Function(arguments='{"city":"Bangalore"}', name='get_weather'), type='function')])]

In [ ]:
# Execute tool to send back observation to MODEL

import json

for call in res.choices[0].message.tool_calls:
  tool_id = call.id
  func_name = call.function.name
  _args = json.loads(call.function.arguments)

  tool_output = TOOL_FUNCTIONS[func_name](**_args) #get_weather(city='Bangalore')

  MESSAGES.append(
      {"role": "tool", "content": str(tool_output) , "tool_call_id": tool_id, "name": func_name}
  )

  print(tool_id, func_name, _args, tool_output)



dcq8w3g8h get_weather {'city': 'Bangalore'} 20


In [ ]:
MESSAGES

[{'role': 'user', 'content': 'How is the weather in Bangalore?'},
 ChatCompletionMessage(content=None, role='assistant', annotations=None, executed_tools=None, function_call=None, reasoning=None, tool_calls=[ChatCompletionMessageToolCall(id='dcq8w3g8h', function=Function(arguments='{"city":"Bangalore"}', name='get_weather'), type='function')]),
 {'role': 'tool',
  'content': '20',
  'tool_call_id': 'dcq8w3g8h',
  'name': 'get_weather'}]

In [ ]:
# Call LLM for Processiing the output
from groq import Groq
client = Groq()
MODEL = "llama-3.1-8b-instant"

res = client.chat.completions.create(
    model = MODEL,
    max_tokens = 1000,
    tools= TOOLS,
    messages= MESSAGES
)




In [ ]:
res.choices[0].message.content

'The current temperature in Bangalore is 20 degrees Celsius.'

In [ ]:
# Tool Calling LOOP
import json

# Question
QUESTION = "Based on weather suggest me an activity in Bengaluru?"

# Create messages
MESSAGES = [{"role": "user", "content": QUESTION}]

# Max Steps
MAX_STEPS = 5

for step in range(MAX_STEPS):
  print(f"==== Step {step} ====")

  # call llm
  from groq import Groq
  client = Groq()
  MODEL = "llama-3.1-8b-instant"

  res = client.chat.completions.create(
      model = MODEL,
      max_tokens = 1000,
      tools= TOOLS,
      messages= MESSAGES,
      parallel_tool_calls=False
  )

  # Append Assistant message
  MESSAGES.append(res.choices[0].message)

  # Break if no tool calls
  if not res.choices[0].message.tool_calls:
    print(f"Final Ouput: {res.choices[0].message.content}")
    break

  # Tool Call
  for call in res.choices[0].message.tool_calls:
    tool_id = call.id
    func_name = call.function.name
    _args = json.loads(call.function.arguments)

    tool_output = TOOL_FUNCTIONS[func_name](**_args) #get_weather(city='Bangalore')

    MESSAGES.append(
        {"role": "tool", "content": str(tool_output) , "tool_call_id": tool_id, "name": func_name}
    )

    print("Tool call -> ", tool_id, func_name, _args, tool_output)




==== Step 0 ====
Tool call ->  1d29j5nrh get_weather {'city': 'Bengaluru'} 20
==== Step 1 ====
Tool call ->  sfnpwfe5r suggest_activity {'temperature': 20} Jump on Bed
==== Step 2 ====
Final Ouput: It's suggested to jump on the bed if you're feeling playful, though this activity depends more on your mood than the temperature.
